In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

data_path = r"C:\Users\hiro2\Documents\EEG_project\data"
df = pd.read_csv(f"{data_path}\\features.csv")

print(f"処理前: {len(df)}件")
print(f"Arousal統計:\n{df['arousal'].describe()}\n")

# IQR法で外れ値検出
Q1 = df['arousal'].quantile(0.25)
Q3 = df['arousal'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 3 * IQR
upper = Q3 + 3 * IQR

outliers = df[(df['arousal'] < lower) | (df['arousal'] > upper)]
print(f"外れ値:\n{outliers[['subject','song_id','arousal']]}\n")

df_clean = df[(df['arousal'] >= lower) & (df['arousal'] <= upper)].copy()
print(f"処理後: {len(df_clean)}件")

# 被験者ごとにArousalをz-score正規化（個人差を吸収）
df_clean['arousal_z'] = df_clean.groupby('subject')['arousal'].transform(
    lambda x: (x - x.mean()) / (x.std() + 1e-10)
)
df_clean['enjoyment_norm'] = df_clean['enjoyment'] / 5.0

# 保存
df_clean.to_csv(f"{data_path}\\features_clean.csv", index=False)
print(f"\nfeatures_clean.csv を保存")

# 可視化
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df['arousal'], bins=30, color='salmon', alpha=0.7)
axes[0].set_title('Arousal（処理前）')
axes[0].set_xlabel('Arousal (β/α)')

axes[1].hist(df_clean['arousal'], bins=30, color='steelblue', alpha=0.7)
axes[1].set_title('Arousal（外れ値除去後）')
axes[1].set_xlabel('Arousal (β/α)')

axes[2].hist(df_clean['arousal_z'], bins=30, color='seagreen', alpha=0.7)
axes[2].set_title('Arousal（z-score正規化後）')
axes[2].set_xlabel('z-score')

plt.tight_layout()
plt.savefig(f"{data_path}\\arousal_distribution.png", dpi=150)
plt.show()
print("グラフ保存完了")